# 12 · Recurrent Neural Networks, LSTM and GRU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anuxlab/deep-learning-systems/blob/main/notebooks/12-rnn-lstm-gru/12_rnn_lstm_gru.ipynb)
[![Run on GitHub Actions](https://img.shields.io/badge/run-GitHub%20Actions-2088FF?logo=githubactions&logoColor=white)](https://github.com/anuxlab/deep-learning-systems/actions/workflows/nb-12-rnn-lstm-gru.yml)

| | |
|---|---|
| **Wiki chapter** | [Module 12](https://github.com/anuxlab/deep-learning-systems/wiki/Module-12) |
| **Level** | Intermediate |
| **Typical CPU runtime** | ~2 min (GitHub free runner, no GPU) |
| **Prerequisites** | Modules 07, 09 |

## What you will do

- Implement a vanilla RNN cell and BPTT, and verify with a gradient check.
- Measure vanishing/exploding gradients across time and the effect of gradient clipping.
- Implement LSTM and GRU cells from equations and match PyTorch's cells exactly.
- Compare RNN/LSTM/GRU on a long-range memory task, and train a character-level language model with sampling.

> **How to use this notebook.** Read each markdown cell, *predict* what the next code cell will print, then run it.
> Every section ends with `check(...)` assertions: if one fails, something in your understanding (or the code) is off.
> Figures are saved to `figures/` and a machine-readable summary to `metrics.json` next to this notebook.

## 0 · Setup and reproducibility

We import shared helpers (`show` saves each figure to `figures/`, `check` asserts and logs, `timer` logs wall-clock time), fix all random seeds and print an environment report so every run is traceable (see the wiki's *Implementation and Research Standards*).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "_common"))   # shared helpers: notebooks/_common/dlsys.py
from dlsys import *                                       # np, plt, torch, seed_everything, show, check, timer, ...

SEED = seed_everything(42)
ENV = env_report()

## 1 · The vanilla RNN and its unrolled graph
$$h_t=\tanh(W_{xh}x_t+W_{hh}h_{t-1}+b_h),\qquad y_t=W_{hy}h_t+b_y.$$
The **same** weights are applied at every time step (parameter sharing), so a sequence of any length is processed by one small model. Backpropagation through time (BPTT) is ordinary backprop on the unrolled graph, summing the gradient contributions of each time step to the shared weights. We write it in NumPy and gradient-check it.

In [ ]:
class NpRNN:
    def __init__(s, nin, nh, nout, seed=0):
        r = np.random.default_rng(seed); s.Wxh = r.normal(0, .4, (nin, nh)); s.Whh = r.normal(0, .4, (nh, nh)); s.bh = np.zeros(nh); s.Why = r.normal(0, .4, (nh, nout)); s.by = np.zeros(nout)
    def params(s): return [s.Wxh, s.Whh, s.bh, s.Why, s.by]
    def loss_grads(s, X, Y):                       # X: (T,nin), Y: (T,nout), loss = 0.5*sum ||y_t - Y_t||^2
        T = len(X); H = [np.zeros(s.bh.shape)]; P = []
        for t in range(T):
            H.append(np.tanh(X[t] @ s.Wxh + H[-1] @ s.Whh + s.bh)); P.append(H[-1] @ s.Why + s.by)
        P = np.array(P); loss = .5 * np.sum((P - Y) ** 2)
        g = [np.zeros_like(p) for p in s.params()]; dh_next = np.zeros_like(s.bh)
        for t in reversed(range(T)):
            dy = P[t] - Y[t]; g[3] += np.outer(H[t + 1], dy); g[4] += dy
            dh = dy @ s.Why.T + dh_next; dz = dh * (1 - H[t + 1] ** 2)          # through tanh
            g[0] += np.outer(X[t], dz); g[1] += np.outer(H[t], dz); g[2] += dz; dh_next = dz @ s.Whh.T   # shared weights accumulate over t
        return loss, g
r = np.random.default_rng(1); net = NpRNN(3, 5, 2); X, Y = r.normal(size=(7, 3)), r.normal(size=(7, 2))
loss, grads = net.loss_grads(X, Y); worst = 0
for p, g in zip(net.params(), grads):
    for idx in np.ndindex(*p.shape):
        old = p[idx]; p[idx] = old + 1e-6; lp = net.loss_grads(X, Y)[0]; p[idx] = old - 1e-6; lm = net.loss_grads(X, Y)[0]; p[idx] = old
        worst = max(worst, abs((lp - lm) / 2e-6 - g[idx]) / (abs(g[idx]) + abs((lp - lm) / 2e-6) + 1e-9))
check(worst < 1e-5, f"BPTT gradients match finite differences (worst relative error {worst:.1e})")

## 2 · Vanishing and exploding gradients through time
$\frac{\partial h_T}{\partial h_1}=\prod_{t=2}^{T}\mathrm{diag}(\phi'_t)\,W_{hh}$. Its norm behaves like $\rho^{T}$ where $\rho$ relates to the spectral radius of $W_{hh}$ (times $\phi'\le1$): $\rho<1$ vanishes, $\rho>1$ explodes. We measure $\|\partial h_T/\partial h_t\|$ with autograd for different spectral radii.

In [ ]:
def jacobian_norms(rho, T=60, n=32, seed=0):
    torch.manual_seed(seed); W = torch.randn(n, n); W = W / torch.linalg.eigvals(W).abs().max() * rho
    h0 = torch.zeros(n, requires_grad=True); xs = .1 * torch.randn(T, n); h = h0; hs = [h0]
    for t in range(T): h = torch.tanh(W @ h + xs[t]); h.retain_grad(); hs.append(h)
    h.sum().backward(); return np.array([hh.grad.norm().item() for hh in hs[1:]])      # d(sum h_T)/d h_t
rhos = [.5, 1.0, 1.5, 3.0]
for rho in rhos: plt.semilogy(jacobian_norms(rho), label=f"ρ(W_hh) = {rho}")
plt.xlabel("time step t"); plt.ylabel("‖∂ sum(h_T)/∂h_t‖"); plt.legend(); plt.title("Gradient flow back through time (T=60)"); show("bptt_gradient_flow")
n05, n3 = jacobian_norms(.5), jacobian_norms(3.0)
check(n05[0] < n05[-1] * 1e-6, "ρ=0.5: gradient w.r.t. early steps vanishes"); check(n3[0] > n3[-1] * 0.01 or n3[0] > 1e-3, "ρ=3: gradient does not vanish (tanh saturation limits explosion)")

## 3 · LSTM and GRU from equations
**LSTM:** $i,f,o=\sigma(\cdot)$, $g=\tanh(\cdot)$; $c_t=f\odot c_{t-1}+i\odot g$; $h_t=o\odot\tanh(c_t)$. The cell state $c$ gets an *additive* update, so $\partial c_t/\partial c_{t-1}=f$ — with $f\approx1$ gradients travel unchanged over long horizons (the "constant error carousel").

**GRU:** $z=\sigma(\cdot)$ (update), $r=\sigma(\cdot)$ (reset); $\tilde h=\tanh(W x+U(r\odot h_{t-1}))$; $h_t=(1-z)\odot h_{t-1}+z\odot\tilde h$ (PyTorch's convention).
We implement both and require **exact agreement** with `nn.LSTMCell` / `nn.GRUCell` after copying weights.

In [ ]:
sig = lambda x: 1 / (1 + np.exp(-x))
def lstm_step(x, h, c, Wih, Whh, bih, bhh):                 # PyTorch gate order: i, f, g, o
    z = Wih @ x + bih + Whh @ h + bhh; H = len(h); i, f, g, o = sig(z[:H]), sig(z[H:2*H]), np.tanh(z[2*H:3*H]), sig(z[3*H:])
    c2 = f * c + i * g; return o * np.tanh(c2), c2
def gru_step(x, h, Wih, Whh, bih, bhh):                      # PyTorch gate order: r, z, n
    H = len(h); gi, gh = Wih @ x + bih, Whh @ h + bhh
    r_, z_ = sig(gi[:H] + gh[:H]), sig(gi[H:2*H] + gh[H:2*H]); n_ = np.tanh(gi[2*H:] + r_ * gh[2*H:]); return (1 - z_) * n_ + z_ * h
torch.manual_seed(0); lc, gc = nn.LSTMCell(4, 6).double(), nn.GRUCell(4, 6).double(); xs = torch.randn(10, 4, dtype=torch.float64)
hT, cT = torch.zeros(6, dtype=torch.float64), torch.zeros(6, dtype=torch.float64); hn, cn, gn, gT = np.zeros(6), np.zeros(6), np.zeros(6), torch.zeros(6, dtype=torch.float64)
for t in range(10):
    hT, cT = lc(xs[t][None], (hT[None], cT[None])); hT, cT = hT[0], cT[0]
    hn, cn = lstm_step(xs[t].numpy(), hn, cn, *[p.detach().numpy() for p in (lc.weight_ih, lc.weight_hh, lc.bias_ih, lc.bias_hh)])
    gT = gc(xs[t][None], gT[None])[0]; gn = gru_step(xs[t].numpy(), gn, *[p.detach().numpy() for p in (gc.weight_ih, gc.weight_hh, gc.bias_ih, gc.bias_hh)])
check(np.allclose(hn, hT.detach().numpy(), atol=1e-12) and np.allclose(cn, cT.detach().numpy(), atol=1e-12), "from-scratch LSTM == nn.LSTMCell over 10 steps")
check(np.allclose(gn, gT.detach().numpy(), atol=1e-12), "from-scratch GRU == nn.GRUCell over 10 steps")
lp, gp_, rp = count_params(nn.LSTM(32, 64)), count_params(nn.GRU(32, 64)), count_params(nn.RNN(32, 64)); print(f"params  RNN {rp:,}  GRU {gp_:,}  LSTM {lp:,}  (ratios 1 : 3 : 4)")
check(abs(gp_ / rp - 3) < .05 and abs(lp / rp - 4) < .05, "parameter ratio RNN:GRU:LSTM = 1:3:4")

## 4 · Gradient flow in gated vs vanilla cells (mechanism)
Before training anything, measure $\|\partial\,\mathrm{out}_T/\partial x_t\|$ for random sequences with PyTorch's default initialisation (LSTM with forget-gate bias set to 1, the common trick that keeps $f\approx0.73$ at the start). This is a *deterministic* diagnostic of how much learning signal reaches early inputs.

In [ ]:
def input_grad_profile(kind, T=100, h=64, trials=20, forget_bias=False):
    out = np.zeros(T)
    for s_ in range(trials):
        torch.manual_seed(s_); m = {"RNN": nn.RNN, "GRU": nn.GRU, "LSTM": nn.LSTM}[kind](8, h, batch_first=True)
        if forget_bias and kind == "LSTM":
            for n_, p_ in m.named_parameters():
                if "bias_ih" in n_: p_.data[h:2*h] = 1.0
        x = torch.randn(1, T, 8, requires_grad=True); o, _ = m(x); o[0, -1].sum().backward(); out += x.grad[0].norm(dim=1).numpy() / trials
    return out
prof = {"RNN": input_grad_profile("RNN"), "GRU": input_grad_profile("GRU"), "LSTM": input_grad_profile("LSTM"), "LSTM (forget bias 1)": input_grad_profile("LSTM", forget_bias=True)}
for k, v in prof.items(): plt.semilogy(np.arange(1, 101), np.maximum(v, 1e-30), label=k)
plt.xlabel("input position t (output read at t=100)"); plt.ylabel("mean ‖∂out_T/∂x_t‖"); plt.legend(); plt.title("How much gradient reaches early inputs?"); show("gated_gradient_flow")
for k, v in prof.items(): print(f"{k:22s} gradient at t=1: {v[0]:.2e}   at t=50: {v[49]:.2e}   at t=100: {v[-1]:.2e}")
check(prof["LSTM (forget bias 1)"][0] > 1e3 * max(prof["RNN"][0], 1e-30), "gated cell with forget-bias 1 passes >1000× more gradient to t=1 than the vanilla RNN")
METRICS["grad_t1"] = {k: float(v[0]) for k, v in prof.items()}

## 5 · A long-range memory task: RNN vs GRU vs LSTM
Task ("first-token recall"): the sequence starts with a class token $a\in\{0..3\}$, followed by $T$ noise tokens; the model must output $a$ at the end (chance = 25%). We train each architecture for several delays $T$ (single seed, Adam, gradient clipping 1.0, LSTM forget-bias 1). **Read the outcome honestly**: with a short training budget, success/failure at a given $T$ depends on optimisation luck as well as architecture — one seed is an anecdote, not a benchmark (see exercises).

In [ ]:
def make_batch(B, T, rng):
    a = rng.integers(0, 4, B); x = np.zeros((B, T + 1, 5), np.float32); x[np.arange(B), 0, a] = 1; x[:, 1:, 4] = rng.normal(size=(B, T)); return torch.tensor(x), torch.tensor(a)
class SeqClf(nn.Module):
    def __init__(s, kind, h=48):
        super().__init__(); s.rnn = {"RNN": nn.RNN, "GRU": nn.GRU, "LSTM": nn.LSTM}[kind](5, h, batch_first=True); s.out = nn.Linear(h, 4)
        if kind == "LSTM":
            for n, p in s.rnn.named_parameters():
                if "bias_ih" in n: p.data[h:2 * h] = 1.0
    def forward(s, x): o, _ = s.rnn(x); return s.out(o[:, -1])
def train_mem(kind, T, steps=1200, seed=0):
    torch.manual_seed(seed); rng = np.random.default_rng(seed); m = SeqClf(kind); o = torch.optim.Adam(m.parameters(), 3e-3)
    for _ in range(steps):
        x, y = make_batch(64, T, rng); o.zero_grad(); F.cross_entropy(m(x), y).backward(); torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); o.step()
    x, y = make_batch(1000, T, np.random.default_rng(999))
    with torch.no_grad(): return (m(x).argmax(1) == y).float().mean().item()
Ts = [10, 40, 100]; acc = {k: [] for k in ["RNN", "GRU", "LSTM"]}
with timer("memory_task"):
    for k in acc:
        for T in Ts: acc[k].append(train_mem(k, T)); print(f"{k:5s} T={T:3d}  accuracy {acc[k][-1]:.3f}")
for k, v in acc.items(): plt.plot(Ts, v, "o-", label=k)
plt.axhline(.25, color="k", ls=":", label="chance"); plt.xlabel("delay T"); plt.ylabel("accuracy"); plt.legend(); plt.title("Long-range memory: first-token recall (1 seed)"); show("memory_task")
check(all(a[0] > .9 for a in acc.values()), "every architecture solves the short-delay task (T=10)")
METRICS["memory_task_acc"] = acc

## 6 · Character-level language modelling with teacher forcing
A char-LM predicts the next character: $p(x_t\mid x_{<t})$. **Teacher forcing** feeds the *true* previous character as input during training (all time steps in one parallel loss); at generation time we feed back our own samples. The corpus is the *Zen of Python* (`import this`, decoded from ROT13 — fully offline). Since the text is tiny we expect the model to memorise it; the point is the mechanics: truncated BPTT, loss in nats/bits per character, sampling with temperature.

In [ ]:
import codecs, this
text = codecs.decode(this.s, "rot13"); chars = sorted(set(text)); stoi = {c: i for i, c in enumerate(chars)}; itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text]); V = len(chars); print(f"corpus: {len(text)} chars, vocabulary {V}")
class CharGRU(nn.Module):
    def __init__(s, V, E=32, H=128): super().__init__(); s.emb = nn.Embedding(V, E); s.rnn = nn.GRU(E, H, num_layers=2, batch_first=True); s.out = nn.Linear(H, V)
    def forward(s, x, h=None): o, h = s.rnn(s.emb(x), h); return s.out(o), h
torch.manual_seed(0); lm = CharGRU(V); o = torch.optim.Adam(lm.parameters(), 5e-3); L = 64; losses = []
with timer("char_lm_training"):
    for step in range(600):
        ix = torch.randint(0, len(data) - L - 1, (32,)); x = torch.stack([data[i:i + L] for i in ix]); y = torch.stack([data[i + 1:i + L + 1] for i in ix])   # y = x shifted by one
        logits, _ = lm(x); loss = F.cross_entropy(logits.reshape(-1, V), y.reshape(-1)); o.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(lm.parameters(), 1.0); o.step(); losses.append(loss.item())
        if step % 100 == 0: print(f"step {step:4d}  loss {loss.item():.3f}  ({loss.item()/np.log(2):.2f} bits/char)")
print(f"uniform baseline: {np.log(V):.3f} nats; final loss {np.mean(losses[-20:]):.3f}")
check(np.mean(losses[-20:]) < .5 * np.log(V), "char-LM loss is less than half of the uniform baseline")
plt.plot(losses); plt.xlabel("step"); plt.ylabel("cross-entropy (nats)"); plt.title("Char-LM training loss"); show("char_lm_loss"); METRICS["char_lm_final_loss"] = float(np.mean(losses[-20:]))
@torch.no_grad()
def sample(lm, start="Beautiful", n=200, temp=1.0):
    x = torch.tensor([[stoi[c] for c in start]]); logits, h = lm(x); out = list(start)
    for _ in range(n):
        p = F.softmax(logits[0, -1] / temp, -1); nxt = torch.multinomial(p, 1).item(); out.append(itos[nxt]); logits, h = lm(torch.tensor([[nxt]]), h)
    return "".join(out)
for T_ in [0.5, 1.0]: print(f"--- temperature {T_} ---\n{sample(lm, temp=T_)}\n")

## 7 · Bidirectional and stacked RNN shapes
A bidirectional RNN concatenates a forward and a backward pass (output feature size $2H$); stacking feeds layer $l$'s outputs to layer $l+1$. Final hidden state shape is `(num_layers × num_directions, B, H)`. Bidirectionality is only valid when the *whole* sequence is available (tagging, encoding) — not for autoregressive generation.

In [ ]:
rnn = nn.LSTM(10, 16, num_layers=3, bidirectional=True, batch_first=True); x = torch.randn(4, 7, 10); out, (h, c) = rnn(x)
print("output:", tuple(out.shape), " h_n:", tuple(h.shape)); check(out.shape == (4, 7, 32) and h.shape == (6, 4, 16), "output (B,T,2H); h_n (layers·dirs, B, H)")

In [ ]:
save_metrics("12", METRICS)
log.info('notebook 12 finished OK')

## Wrap-up

## Exercises (do these next)

1. Repeat the memory task over 5 seeds and report mean ± std per architecture and delay; do gated cells win reliably at T=100?
2. Implement truncated BPTT with a carried hidden state across chunks in the char-LM.
3. Train the char-LM with an LSTM; compare bits/char at equal parameter count.

## Where to read more

- Wiki chapter: [Module 12](https://github.com/anuxlab/deep-learning-systems/wiki/Module-12) (derivations, worked example, Q&A).
- Elman (1990); Hochreiter & Schmidhuber (1997) LSTM; Cho et al. (2014) GRU; Pascanu et al. (2013) *On the difficulty of training RNNs*; Karpathy, *The unreasonable effectiveness of RNNs*.